# Grupo 1 — validação sanitária da base Bitcoin

Este notebook usa o módulo comum do projeto para verificar nulos, duplicatas, datas e regularidade diária. Nenhuma correção é aplicada automaticamente à base bruta.

In [1]:
from pathlib import Path
import sys
import pandas as pd

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'validacao_bases.py').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('Não foi possível localizar validacao_bases.py.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from validacao_bases import (
    CONFIGURACOES, calcular_sha256, carregar_base, extrair_datas,
    relatorios_como_dataframe, validar_base,
)

NOME_BASE = 'bitcoin'
config = CONFIGURACOES[NOME_BASE]
dados = carregar_base(NOME_BASE, RAIZ)
print(f'Base: {NOME_BASE} | formato: {dados.shape[0]:,} linhas x {dados.shape[1]} colunas')

C:\Users\gabrieloliveira-ieg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Base: bitcoin | formato: 2,922 linhas x 9 colunas


In [2]:
display(dados.head())
display(dados.dtypes.rename('tipo').to_frame())

,timeOpen,timeClose,timeHigh,timeLow,priceOpen,priceHigh,priceLow,priceClose,volume
0,1789300800000,1789387199999,1789377900000,1789341240000,77270.180637,77415.656252,76498.384179,76838.155243,1.360787e+10
1,1789214400000,1789300799999,1789268040000,1789285260000,77175.071855,77479.658602,77045.002358,77270.465275,1.266477e+10
2,1789128000000,1789214399999,1789178640000,1789173060000,76559.124754,79818.337264,76162.917834,77173.793101,3.741100e+10
3,1789041600000,1789127999999,1789063140000,1789125360000,78260.386707,78520.968768,76470.640041,76568.124134,3.012011e+10
4,1788955200000,1789041599999,1788986220000,1789035000000,78440.597811,79737.300200,77768.148072,78259.519666,2.919241e+10


,tipo
timeOpen,int64
timeClose,int64
timeHigh,int64
timeLow,int64
priceOpen,float64
priceHigh,float64
priceLow,float64
priceClose,float64
volume,float64


## Resultado consolidado

A frequência esperada é diária (`D`). O critério `aprovada` é estrito: exige ausência de nulos, duplicatas, datas inválidas/repetidas e lacunas temporais.

In [3]:
relatorio = validar_base(NOME_BASE, RAIZ)
display(relatorios_como_dataframe({NOME_BASE: relatorio}))
display(pd.Series(relatorio.nulos_por_coluna, name='quantidade_de_nulos').to_frame())

C:\Users\gabrieloliveira-ieg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,nome,linhas,colunas,linhas_com_nulos,duplicatas_exatas,datas_invalidas,datas_duplicadas,ordenacao_datas,frequencia_esperada,frequencia_regular,timestamps_ausentes,timestamps_fora_da_grade,aprovada
0,bitcoin,2922,9,0,0,0,0,decrescente,D,False,23,0,False


,quantidade_de_nulos


In [4]:
datas = extrair_datas(dados, config)
problemas = dados.loc[dados.duplicated(keep=False) | datas.duplicated(keep=False)].copy()
problemas.insert(0, 'data_normalizada', datas.loc[problemas.index])
print(f'Linhas com duplicidade exata ou temporal: {len(problemas):,}')
display(problemas.head(10))
print('Exemplos de datas ausentes:', relatorio.exemplos_timestamps_ausentes)
print('Ordenação encontrada:', relatorio.ordenacao_datas)

Linhas com duplicidade exata ou temporal: 0


,data_normalizada,timeOpen,timeClose,timeHigh,timeLow,priceOpen,priceHigh,priceLow,priceClose,volume


Exemplos de datas ausentes: ['2023-09-18T12:00:00+00:00', '2023-09-19T12:00:00+00:00', '2023-09-20T12:00:00+00:00', '2023-09-21T12:00:00+00:00', '2023-09-23T12:00:00+00:00']
Ordenação encontrada: decrescente


## Evidência para o congelamento

O hash abaixo identifica exatamente o arquivo analisado. Para congelar as cinco bases de uma vez, execute `congelar_bases('dados_congelados/v1')` uma única vez na raiz do projeto.

In [5]:
arquivo = RAIZ / config.caminho
print('Arquivo:', arquivo.relative_to(RAIZ))
print('SHA-256:', calcular_sha256(arquivo))
print('Conclusão:', 'APROVADA' if relatorio.aprovada else 'REQUER TRATAMENTO ANTES DA MODELAGEM')

Arquivo: grupo1\bitcoin.xlsx
SHA-256: 80d44fcd7a8053297aa72b91348128754046408b9146bfc6a4a6c5ddde826a9f
Conclusão: REQUER TRATAMENTO ANTES DA MODELAGEM
